# 14 端侧 Agent 与新范式

## 趋势

从单轮问答 → **多轮 Agent**：工具调用、规划、状态管理，尽量全链路离线。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")

import json
import re

## 14.1 最小 Function Calling + 语法约束

In [ ]:
TOOLS = {
    "get_battery": {"desc": "读取电量百分比", "args": {}},
    "set_alarm": {"desc": "设置闹钟", "args": {"time": "HH:MM", "label": "str"}},
}


def build_system_prompt(tools=TOOLS) -> str:
    catalog = json.dumps(tools, ensure_ascii=False)
    return (
        "你是端侧助手。需要调用工具时，只输出一行 JSON:\n"
        '{"tool": name, "arguments": {...}}\n'
        f"可用工具: {catalog}"
    )


def parse_tool_call(text: str):
    """从文本中提取第一个可解析且工具合法的 JSON 对象。"""
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(text[i:])
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and obj.get("tool") in TOOLS:
            return obj
    return None
samples = [
    '好的，帮你查电量。{"tool": "get_battery", "arguments": {}}',
    "明天叫我起床",  # 无工具
    '{"tool": "hack", "arguments": {}}',  # 非法工具
]
print(build_system_prompt())
print("=== 解析 ===")
for s in samples:
    print(s, "→", parse_tool_call(s))

## 14.2 端侧 Agent 循环（规划 → 工具 → 观察）

In [ ]:
class OnDeviceAgent:
    def __init__(self, max_steps=4):
        self.max_steps = max_steps
        self.state = {"battery": 67}

    def llm(self, goal: str, obs: str) -> str:
        # 教学用规则假 LLM：按已收集观察决定下一步
        need_bat = ("电量" in goal) or ("battery" in goal)
        need_alarm = ("闹钟" in goal) or ("alarm" in goal)
        if need_bat and ("battery=" not in obs):
            return '{"tool": "get_battery", "arguments": {}}'
        if need_alarm and ("alarm_set=" not in obs):
            return '{"tool": "set_alarm", "arguments": {"time": "07:00", "label": "起床"}}'
        return f"完成：基于观察 [{obs}] 已处理「{goal}」"

    def run_tool(self, call: dict) -> str:
        if call["tool"] == "get_battery":
            return f"battery={self.state['battery']}%"
        if call["tool"] == "set_alarm":
            self.state["alarm"] = call["arguments"]
            return f"alarm_set={call['arguments']}"
        return "error"

    def run(self, goal: str) -> str:
        obs_parts = []
        for step in range(self.max_steps):
            obs = " | ".join(obs_parts)
            out = self.llm(goal, obs)
            call = parse_tool_call(out)
            if call is None:
                return out
            tip = self.run_tool(call)
            obs_parts.append(tip)
            print(f"step{step}: call={call} obs={tip}")
        return "达到步数上限"


agent = OnDeviceAgent()
print(agent.run("查看电量并设置闹钟"))

## 14.3 端侧 RAG 片段检索（示意）

In [ ]:
docs = [
    "出差报销需要发票和行程单",
    "端侧模型推荐 INT4 量化",
    "电池低于 20% 应切换小模型",
]


def embed(text: str) -> torch.Tensor:
    # 极简 bag-of-chars 哈希嵌入（演示）
    v = torch.zeros(64)
    for ch in text:
        v[hash(ch) % 64] += 1.0
    return F.normalize(v, dim=0)


def retrieve(query: str, k=2):
    q = embed(query)
    scores = [float(embed(d) @ q) for d in docs]
    idx = np.argsort(scores)[::-1][:k]
    return [(docs[i], scores[i]) for i in idx]


print(retrieve("手机电量低怎么部署"))

## 小结

1. 工具调用必须 **文法/JSON 约束**，避免重试浪费 token。
2. Agent 要限制步数与内存（KV 增长 → MLA/TurboQuant）。
3. RAG + Agent 可全离线：本地向量库 + 本地 LLM + 本地工具。